# Plot 1

In [6]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import gc
import glob
import os
import re

print(os.getcwd())

meta_df = pd.read_parquet(
     "003_fnr_value_learning_extended/output/dashboard/meta_info.parquet"
)

files = glob.glob(
    "003_fnr_value_learning_extended/output/dashboard/003_fnr_value_learning_*.parquet"
)

# Determine the available iterations by filename
# Files must follow this convention: 003_value_learning_i.parquet
n_iterations = 0
iterations = []
for f in files:
    match = re.search(r"_(\d+)\.parquet$", f)

    if match:
        iterations += [int(match.group(1))]

if iterations:
    n_iterations = len(iterations)

lambda_A_vals = meta_df["lambda_A_vals"].iloc[0]
eta_vals = meta_df["eta_vals"].iloc[0]
gamma_vals = meta_df["gamma_vals"].iloc[0]
C_vals = meta_df["C_vals"].iloc[0]
alpha_vals = meta_df["alpha_vals"].iloc[0]
kappa_vals = meta_df["kappa_vals"].iloc[0]

del meta_df
gc.collect()

# Values to plot
iteration = "expected"
lambda_A = lambda_A_vals[3]
lambda_A2 = lambda_A_vals[6]
lambda_A3 = lambda_A_vals[8]
eta = eta_vals[1] # 0.111
gamma = 0
C = 1
alpha = 1
alpha2 = alpha_vals[4]
kappa = 1
kappa2 = kappa_vals[2]

print(lambda_A, eta, gamma, C, alpha, kappa)

def get_df_by_params(lambda_A, eta, gamma, C, alpha, kappa):
    filters = [
        ("lambda_A", "=", lambda_A),
        ("eta", "=", eta),
        ("gamma", "=", gamma),
        ("C", "=", C),
        ("alpha", "=", alpha),
        ("kappa", "=", kappa)
    ]

    # read only filtered rows and needed columns
    cols_needed = ["Step", "V_mean", "V_std", "M_A_mean", "M_A_std"]

    table = pq.read_table(
            "003_fnr_value_learning_extended/output/dashboard/003_fnr_value_learning_summary.parquet",
        columns=cols_needed,
        filters=filters
    )

    dff = table.to_pandas().sort_values("Step")

    return dff


fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=("a", "b", "c", "d"),
)

# ------------------------------ Plot A --------------------

dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       C=C,
                       alpha=alpha,
                       kappa=kappa)

# shaded bounds
V_upper = dff["V_mean"] + dff["V_std"]
V_lower = dff["V_mean"] - dff["V_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["V_mean"],
        mode="lines",
        name="Value of state",
        line=dict(color="blue"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([V_upper, V_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(0,0,255,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_hline(y=0, row=1, col=1)

# ------------------------------ Plot B --------------------
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=2,
    col=1,
)


fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=2,
    col=1,
)

fig.add_hline(y=0, row=2, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(title_text="Value of state", row=1, col=1)
fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.4, 0.4],
    row=2,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=0.9,
    y=1,
    showarrow=False,
    text=rf"$\lambda^{{(A)}} = {lambda_A:.3f}$",
    font=dict(size=18),
    row=2,
    col=1,
)

# ------------------------------ Plot C --------------------
dff = get_df_by_params(lambda_A=lambda_A2,
                       eta=eta,
                       gamma=gamma,
                       C=C,
                       alpha=alpha,
                       kappa=kappa)

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=0.9,
    y=1,
    showarrow=False,
    text=rf"$\lambda^{{(A)}} = {lambda_A2:.3f}$",
    font=dict(size=18),
    row=3,
    col=1,
)


fig.add_hline(y=0, row=3, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.4, 0.4],
    row=3,
    col=1,
)

# ------------------------------ Plot D --------------------

dff = get_df_by_params(lambda_A=lambda_A3,
                       eta=eta,
                       gamma=gamma,
                       C=C,
                       alpha=alpha,
                       kappa=kappa)

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=4,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=4,
    col=1,
)

fig.add_hline(y=0, row=4, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.4, 0.4],
    row=4,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=0.9,
    y=1,
    showarrow=False,
    text=rf"$\lambda^{{(A)}} = {lambda_A3:.3f}$",
    font=dict(size=18),
    row=4,
    col=1,
)

fig.add_hline(y=0, row=4, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_xaxes(title_text="Step", row=4, col=1)

# ################

fig.update_annotations(xshift=-360, yshift=-20)
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=50),
    height=900,
    showlegend=False,
)

fig.update_xaxes(
    tickmode="linear",
    dtick=25,
)

fig.write_image("plots/plot1.pdf")

/home/soelderer/projects/formal_model_irritability/simulations
0.33333334 0.11111111 0 1 1 1


# Plot 2

In [7]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import gc
import glob
import os
import re

print(os.getcwd())

meta_df = pd.read_parquet(
     "003_fnr_value_learning_extended/output/dashboard/meta_info.parquet"
)

files = glob.glob(
    "003_fnr_value_learning_extended/output/dashboard/003_fnr_value_learning_*.parquet"
)

# Determine the available iterations by filename
# Files must follow this convention: 003_value_learning_i.parquet
n_iterations = 0
iterations = []
for f in files:
    match = re.search(r"_(\d+)\.parquet$", f)

    if match:
        iterations += [int(match.group(1))]

if iterations:
    n_iterations = len(iterations)

lambda_A_vals = meta_df["lambda_A_vals"].iloc[0]
eta_vals = meta_df["eta_vals"].iloc[0]
gamma_vals = meta_df["gamma_vals"].iloc[0]
C_vals = meta_df["C_vals"].iloc[0]
alpha_vals = meta_df["alpha_vals"].iloc[0]
kappa_vals = meta_df["kappa_vals"].iloc[0]

del meta_df
gc.collect()

# Values to plot
iteration = "expected"
lambda_A = lambda_A_vals[8]
eta = eta_vals[1] # 0.111
gamma = 0
C = 1
alpha = 1
alpha1 = alpha_vals[4]
alpha2 = 0
kappa = 1


def get_df_by_params(lambda_A, eta, gamma, C, alpha, kappa):
    filters = [
        ("lambda_A", "=", lambda_A),
        ("eta", "=", eta),
        ("gamma", "=", gamma),
        ("C", "=", C),
        ("alpha", "=", alpha),
        ("kappa", "=", kappa)
    ]

    # read only filtered rows and needed columns
    cols_needed = ["Step", "V_mean", "V_std", "M_A_mean", "M_A_std"]

    table = pq.read_table(
            "003_fnr_value_learning_extended/output/dashboard/003_fnr_value_learning_summary.parquet",
        columns=cols_needed,
        filters=filters
    )

    dff = table.to_pandas().sort_values("Step")

    return dff


fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=("a", "b", "c", "d"),
)

# ------------------------------ Plot A --------------------

dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       C=C,
                       alpha=alpha,
                       kappa=kappa)

# shaded bounds
V_upper = dff["V_mean"] + dff["V_std"]
V_lower = dff["V_mean"] - dff["V_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["V_mean"],
        mode="lines",
        name="Value of state",
        line=dict(color="blue"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([V_upper, V_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(0,0,255,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_hline(y=0, row=1, col=1)

# ------------------------------ Plot B --------------------
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=2,
    col=1,
)


fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=2,
    col=1,
)

fig.add_hline(y=0, row=2, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(title_text="Value of state", row=1, col=1)
fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.5, 0.5],
    row=2,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=0.9,
    y=1,
    showarrow=False,
    text=rf"$\alpha = {alpha:.3f}$",
    font=dict(size=18),
    row=2,
    col=1,
)

# ------------------------------ Plot C --------------------
dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       C=C,
                       alpha=alpha1,
                       kappa=kappa)

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=0.9,
    y=1,
    showarrow=False,
    text=rf"$\alpha = {alpha1:.3f}$",
    font=dict(size=18),
    row=3,
    col=1,
)


fig.add_hline(y=0, row=3, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.5, 0.5],
    row=3,
    col=1,
)

# ------------------------------ Plot D --------------------

dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       C=C,
                       alpha=alpha2,
                       kappa=kappa)

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=4,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=4,
    col=1,
)

fig.add_hline(y=0, row=4, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.5, 0.5],
    row=4,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=0.9,
    y=1,
    showarrow=False,
    text=rf"$\alpha = {alpha2:.3f}$",
    font=dict(size=18),
    row=4,
    col=1,
)

fig.add_hline(y=0, row=4, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_xaxes(title_text="Step", row=4, col=1)

# ################

fig.update_annotations(xshift=-360, yshift=-20)
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=50),
    height=900,
    showlegend=False,
)

fig.update_xaxes(
    tickmode="linear",
    dtick=25,
)

fig.write_image("plots/plot2.pdf")

/home/soelderer/projects/formal_model_irritability/simulations


# Plot 3

In [8]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import gc
import glob
import os
import re

print(os.getcwd())

meta_df = pd.read_parquet(
     "003_fnr_value_learning_extended/output/dashboard/meta_info.parquet"
)

files = glob.glob(
    "003_fnr_value_learning_extended/output/dashboard/003_fnr_value_learning_*.parquet"
)

# Determine the available iterations by filename
# Files must follow this convention: 003_value_learning_i.parquet
n_iterations = 0
iterations = []
for f in files:
    match = re.search(r"_(\d+)\.parquet$", f)

    if match:
        iterations += [int(match.group(1))]

if iterations:
    n_iterations = len(iterations)

lambda_A_vals = meta_df["lambda_A_vals"].iloc[0]
eta_vals = meta_df["eta_vals"].iloc[0]
gamma_vals = meta_df["gamma_vals"].iloc[0]
C_vals = meta_df["C_vals"].iloc[0]
alpha_vals = meta_df["alpha_vals"].iloc[0]
kappa_vals = meta_df["kappa_vals"].iloc[0]

del meta_df
gc.collect()

# Values to plot
iteration = "expected"
lambda_A = lambda_A_vals[8]
eta = eta_vals[1] # 0.111
gamma = 0
C = 1
alpha = alpha_vals[5]
kappa = 1
kappa1 = kappa_vals[1]
kappa2 = kappa_vals[2]


def get_df_by_params(lambda_A, eta, gamma, C, alpha, kappa):
    filters = [
        ("lambda_A", "=", lambda_A),
        ("eta", "=", eta),
        ("gamma", "=", gamma),
        ("C", "=", C),
        ("alpha", "=", alpha),
        ("kappa", "=", kappa)
    ]

    # read only filtered rows and needed columns
    cols_needed = ["Step", "V_mean", "V_std", "M_A_mean", "M_A_std"]

    table = pq.read_table(
            "003_fnr_value_learning_extended/output/dashboard/003_fnr_value_learning_summary.parquet",
        columns=cols_needed,
        filters=filters
    )

    dff = table.to_pandas().sort_values("Step")

    return dff


fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=("a", "b", "c", "d"),
)

# ------------------------------ Plot A --------------------

dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       C=C,
                       alpha=alpha,
                       kappa=kappa)

# shaded bounds
V_upper = dff["V_mean"] + dff["V_std"]
V_lower = dff["V_mean"] - dff["V_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["V_mean"],
        mode="lines",
        name="Value of state",
        line=dict(color="blue"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([V_upper, V_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(0,0,255,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_hline(y=0, row=1, col=1)

# ------------------------------ Plot B --------------------
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=2,
    col=1,
)


fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=2,
    col=1,
)

fig.add_hline(y=0, row=2, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(title_text="Value of state", row=1, col=1)
fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.5, 0.5],
    row=2,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=1.3,
    y=1,
    showarrow=False,
    text=rf"$\kappa = {kappa:.3f}$",
    font=dict(size=18),
    row=2,
    col=1,
)

# ------------------------------ Plot C --------------------
dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       C=C,
                       alpha=alpha,
                       kappa=kappa1)

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=1.3,
    y=1,
    showarrow=False,
    text=rf"$\kappa = {kappa1:.3f}$",
    font=dict(size=18),
    row=3,
    col=1,
)


fig.add_hline(y=0, row=3, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.5, 0.5],
    row=3,
    col=1,
)

# ------------------------------ Plot D --------------------

dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       C=C,
                       alpha=alpha,
                       kappa=kappa2)

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=4,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=4,
    col=1,
)

fig.add_hline(y=0, row=4, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(
    title_text="Frustration/anger",
    range=[-0.5, 0.5],
    row=4,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=1.3,
    y=1,
    showarrow=False,
    text=rf"$\kappa = {kappa2:.3f}$",
    font=dict(size=18),
    row=4,
    col=1,
)

fig.add_hline(y=0, row=4, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_xaxes(title_text="Step", row=4, col=1)

# ################

fig.update_annotations(xshift=-360, yshift=-20)
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=50),
    height=900,
    showlegend=False,
)

fig.update_xaxes(
    tickmode="linear",
    dtick=25,
)

fig.write_image("plots/plot3.pdf")

/home/soelderer/projects/formal_model_irritability/simulations


# Plot 4

In [74]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import gc
import glob
import os
import re

print(os.getcwd())

meta_df = pd.read_parquet(
     "004_sadness/output/dashboard/meta_info.parquet"
)

files = glob.glob(
    "004_sadness/output/dashboard/004_sadness_*.parquet"
)

# Determine the available iterations by filename
# Files must follow this convention: 003_value_learning_i.parquet
n_iterations = 0
iterations = []
for f in files:
    match = re.search(r"_(\d+)\.parquet$", f)

    if match:
        iterations += [int(match.group(1))]

if iterations:
    n_iterations = len(iterations)

lambda_A_vals = meta_df["lambda_A_vals"].iloc[0]
eta_vals = meta_df["eta_vals"].iloc[0]
gamma_vals = meta_df["gamma_vals"].iloc[0]
alpha_vals = meta_df["alpha_vals"].iloc[0]
kappa_vals = meta_df["kappa_vals"].iloc[0]
lambda_C_vals = meta_df["lambda_C_vals"].iloc[0]
midpoint_C_vals = meta_df["midpoint_vals"].iloc[0]

del meta_df
gc.collect()

# Values to plot
iteration = "expected"
lambda_A = lambda_A_vals[7]
eta = eta_vals[1] # 0.111
gamma = 0
alpha = alpha_vals[2]
kappa = kappa_vals[1]
lambda_C = lambda_C_vals[1]
midpoint_C = midpoint_C_vals[0]
midpoint_C2 = midpoint_C_vals[1]

print(f"lambda_A = {lambda_A}, eta = {eta}, alpha = {alpha}, kappa = {kappa}, lambda_C = {lambda_C}, midpoint_C = {midpoint_C}")


def get_df_by_params(lambda_A, eta, gamma, alpha, kappa, lambda_C,
                     midpoint_C):
    filters = [
            ("lambda_A",
             "=", lambda_A),
            ("eta",
             "=", eta),
            ("gamma",
             "=", gamma),
            ("alpha",
             "=", alpha),
            ("kappa",
             "=", kappa),
            ("lambda_C",
             "=", lambda_C),
            ("midpoint",
             "=", midpoint_C),
    ]

    # read only filtered rows and needed columns
    cols_needed = ["Step", "V_mean", "V_std", "M_A_mean", "M_A_std",
                    "M_S_mean", "M_S_std", "C"]

    table = pq.read_table(
            "004_sadness/output/dashboard/004_sadness_summary.parquet",
        columns=cols_needed,
        filters=filters
    )

    dff = table.to_pandas().sort_values("Step")

    return dff


fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=("a", "b", "c"),
)

# ------------------------------ Plot A --------------------

dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       alpha=alpha,
                       kappa=kappa,
                       lambda_C=lambda_C,
                       midpoint_C=midpoint_C)

# shaded bounds
V_upper = dff["V_mean"] + dff["V_std"]
V_lower = dff["V_mean"] - dff["V_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["V_mean"],
        mode="lines",
        name="Value of state",
        line=dict(color="blue"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([V_upper, V_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(0,0,255,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_hline(y=0, row=1, col=1)

# ------------------------------ Plot B --------------------
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_S_mean"],
        mode="lines",
        name="Sadness",
        line=dict(color="orange"),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_S_upper, M_S_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,165,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["C"],
        mode="lines",
        name="Controllability",
        line=dict(color="green"),
    ),
    row=2,
    col=1,
)

fig.add_hline(y=0, row=2, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(title_text="Value of state", row=1, col=1)
fig.update_yaxes(
    title_text="Emotions",
    range=[-0.5, 1.2],
    row=2,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=1.4,
    y=0.9,
    showarrow=False,
    text=rf"$t_{{mid}}^{{(C)}} = {midpoint_C}$",
    font=dict(size=18),
    row=2,
    col=1,
)

# ------------------------------ Plot C --------------------
dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       alpha=alpha,
                       kappa=kappa,
                       lambda_C=lambda_C,
                       midpoint_C=midpoint_C2)

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_S_mean"],
        mode="lines",
        name="Sadness",
        line=dict(color="orange"),
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_S_upper, M_S_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,165,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_hline(y=0, row=2, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["C"],
        mode="lines",
        name="Controllability",
        line=dict(color="green"),
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_hline(y=0, row=3, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(
    title_text="Emotions",
    range=[-0.5, 1.2],
    row=3,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=1.4,
    y=0.9,
    showarrow=False,
    text=rf"$t_{{mid}}^{{(C)}} = {midpoint_C2}$",
    font=dict(size=18),
    row=3,
    col=1,
)

fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.1,      # move below plots
        xanchor="center",
        x=0.5,
    ),
    margin=dict(t=20, r=10)
)

fig.update_annotations(xshift=-340, yshift=-20)

fig.update_xaxes(
    tickmode="linear",
    dtick=25,
)

fig.update_xaxes(
    title_text="Step",
    row=3,
    col=1
)

fig.write_image("plots/plot4.pdf")
# save ....

/home/soelderer/projects/formal_model_irritability/simulations
lambda_A = 0.8888888955116272, eta = 0.1111111119389534, alpha = 0.4444444477558136, kappa = 1.8888888359069824, lambda_C = 0.25, midpoint_C = 125


# Plot 5

In [80]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import gc
import glob
import os
import re

meta_df = pd.read_parquet(
     "005_response_vigor/output/dashboard/meta_info.parquet"
)

files = glob.glob(
    "005_response_vigor/output/dashboard/005_response_vigor_*.parquet"
)

# Determine the available iterations by filename
# Files must follow this convention: 003_value_learning_i.parquet
n_iterations = 0
iterations = []
for f in files:
    match = re.search(r"_(\d+)\.parquet$", f)

    if match:
        iterations += [int(match.group(1))]

if iterations:
    n_iterations = len(iterations)

lambda_A_vals = meta_df["lambda_A_vals"].iloc[0]
eta_vals = meta_df["eta_vals"].iloc[0]
gamma_vals = meta_df["gamma_vals"].iloc[0]
alpha_vals = meta_df["alpha_vals"].iloc[0]
kappa_vals = meta_df["kappa_vals"].iloc[0]
lambda_C_vals = meta_df["lambda_C_vals"].iloc[0]
midpoint_C_vals = meta_df["midpoint_vals"].iloc[0]
w_v_A_vals = meta_df["w_v_A_vals"].iloc[0]

del meta_df
gc.collect()

# Values to plot
iteration = "expected"
lambda_A = lambda_A_vals[2]  # 0.888
eta = eta_vals[1] # 0.111
gamma = 0
alpha = alpha_vals[2]
kappa = kappa_vals[1]
lambda_C = lambda_C_vals[1]
midpoint_C = midpoint_C_vals[0]
w_v_A = w_v_A_vals[4]
w_v_A2 = w_v_A_vals[8]

print(f"lambda_A = {lambda_A}, eta = {eta}, alpha = {alpha}, kappa = {kappa}, lambda_C = {lambda_C}, midpoint_C = {midpoint_C}")


def get_df_by_params(lambda_A, eta, gamma, alpha, kappa, lambda_C,
                     midpoint_C, w_v_A):
    filters = [
            ("lambda_A",
             "=", lambda_A),
            ("eta",
             "=", eta),
            ("gamma",
             "=", gamma),
            ("alpha",
             "=", alpha),
            ("kappa",
             "=", kappa),
            ("lambda_C",
             "=", lambda_C),
            ("midpoint",
             "=", midpoint_C),
            ("w_v_A",
             "=", w_v_A),
    ]

    # read only filtered rows and needed columns
    cols_needed = ["Step", "V_mean", "V_std", "M_A_mean", "M_A_std",
                    "M_S_mean", "M_S_std", "C", "v_mean", "v_std"]

    table = pq.read_table(
            "005_response_vigor/output/dashboard/005_response_vigor_summary.parquet",
        columns=cols_needed,
        filters=filters
    )

    dff = table.to_pandas().sort_values("Step")

    return dff


fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=("a", "b", "c"),
)

# ------------------------------ Plot A --------------------
dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       alpha=alpha,
                       kappa=kappa,
                       lambda_C=lambda_C,
                       midpoint_C=midpoint_C,
                       w_v_A=w_v_A)

M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]
v_upper = dff["v_mean"] + dff["v_std"]
v_lower = dff["v_mean"] - dff["v_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_S_mean"],
        mode="lines",
        name="Sadness",
        line=dict(color="orange"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_S_upper, M_S_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,165,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["C"],
        mode="lines",
        name="Controllability",
        line=dict(color="green"),
    ),
    row=1,
    col=1,
)

fig.add_hline(y=0, row=1, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(title_text="Value of state", row=1, col=1)
fig.update_yaxes(
    title_text="Emotions",
    range=[-0.5, 1.2],
    row=1,
    col=1,
)

# fig.add_shape(
#     type="line",
#     x0=133.33,
#     x1=133.33,
#     y0=0.5,
#     y1=-0.5,
#     line=dict(dash="dot"),
#     row=1,
#     col=1,
# )

# ------------------------------ Plot B --------------------

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]
v_upper = dff["v_mean"] + dff["v_std"]
v_lower = dff["v_mean"] - dff["v_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["v_mean"],
        mode="lines",
        name="Response vigor",
        line=dict(color="magenta"),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([v_upper, v_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,255,0.2)",
        line=dict(color="rgba(255,0,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.update_yaxes(
    title_text="Response vigor",
    range=[0, 1],
    row=2,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=1.5,
    y=0.9,
    showarrow=False,
    text=rf"$w_{{v,A}} = {w_v_A:.3f}$",
    font=dict(size=18),
    row=2,
    col=1,
)

fig.add_hline(y=0.5, row=2, col=1, line_dash="dot")
fig.add_vline(x=100, line_dash="dash")

# ------------------------------ Plot C --------------------
dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       alpha=alpha,
                       kappa=kappa,
                       lambda_C=lambda_C,
                       midpoint_C=midpoint_C,
                       w_v_A=w_v_A2)

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]
v_upper = dff["v_mean"] + dff["v_std"]
v_lower = dff["v_mean"] - dff["v_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["v_mean"],
        mode="lines",
        name="Response vigor",
        line=dict(color="magenta"),
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([v_upper, v_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,255,0.2)",
        line=dict(color="rgba(255,0,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.update_yaxes(
    title_text="Response vigor",
    range=[0, 1],
    row=3,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=1.5,
    y=0.9,
    showarrow=False,
    text=rf"$w_{{v,A}} = {w_v_A2:.3f}$",
    font=dict(size=18),
    row=3,
    col=1,
)

fig.add_hline(y=0.5, row=3, col=1, line_dash="dot")
fig.add_vline(x=100, line_dash="dash")

fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.1,      # move below plots
        xanchor="center",
        x=0.5,
    ),
    margin=dict(t=20, r=10)
)

fig.update_annotations(xshift=-340, yshift=-5)

fig.update_xaxes(
    tickmode="linear",
    dtick=25,
)

fig.update_xaxes(
    title_text="Step",
    row=3,
    col=1
)

fig.write_image("plots/plot5.pdf")
# save ....

lambda_A = 0.8888888955116272, eta = 0.1111111119389534, alpha = 0.4444444477558136, kappa = 1.8888888359069824, lambda_C = 0.25, midpoint_C = 125


# Plot 6

In [82]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import gc
import glob
import os
import re

meta_df = pd.read_parquet(
     "006_response_inhibition/output/dashboard/meta_info.parquet"
)

files = glob.glob(
    "006_response_inhibition/output/dashboard/006_response_inhibition_*.parquet"
)

# Determine the available iterations by filename
# Files must follow this convention: 003_value_learning_i.parquet
n_iterations = 0
iterations = []
for f in files:
    match = re.search(r"_(\d+)\.parquet$", f)

    if match:
        iterations += [int(match.group(1))]

if iterations:
    n_iterations = len(iterations)

lambda_A_vals = meta_df["lambda_A_vals"].iloc[0]
eta_vals = meta_df["eta_vals"].iloc[0]
gamma_vals = meta_df["gamma_vals"].iloc[0]
alpha_vals = meta_df["alpha_vals"].iloc[0]
kappa_vals = meta_df["kappa_vals"].iloc[0]
lambda_C_vals = meta_df["lambda_C_vals"].iloc[0]
midpoint_C_vals = meta_df["midpoint_vals"].iloc[0]
w_v_A_vals = meta_df["w_v_A_vals"].iloc[0]
I_vals = meta_df["I_vals"].iloc[0]

del meta_df
gc.collect()

# Values to plot
iteration = "expected"
lambda_A = lambda_A_vals[1]  # 0.888
eta = eta_vals[0] # 0.111
gamma = 0
alpha = alpha_vals[1]  # 0.444
kappa = kappa_vals[1]  # 1.888
lambda_C = lambda_C_vals[1]
midpoint_C = midpoint_C_vals[0]
w_v_A = w_v_A_vals[8]
I = 0
I2 = 0.5
I3 = 0.75


print(f"lambda_A = {lambda_A}, eta = {eta}, alpha = {alpha}, kappa = {kappa}, lambda_C = {lambda_C}, midpoint_C = {midpoint_C}, w_v_A = {w_v_A}")


def get_df_by_params(lambda_A, eta, gamma, alpha, kappa, lambda_C,
                     midpoint_C, w_v_A, I):
    filters = [
            ("lambda_A",
             "=", lambda_A),
            ("eta",
             "=", eta),
            ("gamma",
             "=", gamma),
            ("alpha",
             "=", alpha),
            ("kappa",
             "=", kappa),
            ("lambda_C",
             "=", lambda_C),
            ("midpoint",
             "=", midpoint_C),
            ("w_v_A",
             "=", w_v_A),
            ("I",
             "=", I),
    ]

    # read only filtered rows and needed columns
    cols_needed = ["Step", "V_mean", "V_std", "M_A_mean", "M_A_std",
                    "M_S_mean", "M_S_std", "C", "v_mean", "v_std"]

    table = pq.read_table(
            "006_response_inhibition/output/dashboard/006_response_inhibition_summary.parquet",
        columns=cols_needed,
        filters=filters
    )

    dff = table.to_pandas().sort_values("Step")

    return dff


fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=("a", "b", "c", "d"),
)

# ------------------------------ Plot A --------------------
dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       alpha=alpha,
                       kappa=kappa,
                       lambda_C=lambda_C,
                       midpoint_C=midpoint_C,
                       w_v_A=w_v_A,
                       I=I)

M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]
v_upper = dff["v_mean"] + dff["v_std"]
v_lower = dff["v_mean"] - dff["v_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_A_upper, M_A_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_S_mean"],
        mode="lines",
        name="Sadness",
        line=dict(color="orange"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([M_S_upper, M_S_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,165,0,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["C"],
        mode="lines",
        name="Controllability",
        line=dict(color="green"),
    ),
    row=1,
    col=1,
)

fig.add_hline(y=0, row=1, col=1)
fig.add_vline(x=100, line_dash="dash")

fig.update_yaxes(title_text="Value of state", row=1, col=1)
fig.update_yaxes(
    title_text="Emotions",
    range=[-0.5, 1.2],
    row=1,
    col=1,
)

# fig.add_shape(
#     type="line",
#     x0=133.33,
#     x1=133.33,
#     y0=0.5,
#     y1=-0.5,
#     line=dict(dash="dot"),
#     row=1,
#     col=1,
# )

# ------------------------------ Plot B --------------------

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]
v_upper = dff["v_mean"] + dff["v_std"]
v_lower = dff["v_mean"] - dff["v_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["v_mean"],
        mode="lines",
        name="Response vigor",
        line=dict(color="magenta"),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([v_upper, v_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,255,0.2)",
        line=dict(color="rgba(255,0,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.update_yaxes(
    title_text="Response vigor",
    range=[0, 1],
    row=2,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=1.5,
    y=0.9,
    showarrow=False,
    text=rf"$I = {I}$",
    font=dict(size=18),
    row=2,
    col=1,
)

fig.add_hline(y=0.5, row=2, col=1, line_dash="dot")
fig.add_vline(x=100, line_dash="dash")

# ------------------------------ Plot C --------------------
dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       alpha=alpha,
                       kappa=kappa,
                       lambda_C=lambda_C,
                       midpoint_C=midpoint_C,
                       w_v_A=w_v_A,
                       I=I2)

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]
v_upper = dff["v_mean"] + dff["v_std"]
v_lower = dff["v_mean"] - dff["v_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["v_mean"],
        mode="lines",
        name="Response vigor",
        line=dict(color="magenta"),
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([v_upper, v_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,255,0.2)",
        line=dict(color="rgba(255,0,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=3,
    col=1,
)

fig.update_yaxes(
    title_text="Response vigor",
    range=[0, 1],
    row=3,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=1.5,
    y=0.9,
    showarrow=False,
    text=rf"$I = {I2}$",
    font=dict(size=18),
    row=3,
    col=1,
)

fig.add_hline(y=0.5, row=3, col=1, line_dash="dot")

# ------------------------------ Plot D --------------------
dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       alpha=alpha,
                       kappa=kappa,
                       lambda_C=lambda_C,
                       midpoint_C=midpoint_C,
                       w_v_A=w_v_A,
                       I=I3)

# shaded bounds
M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]
v_upper = dff["v_mean"] + dff["v_std"]
v_lower = dff["v_mean"] - dff["v_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["v_mean"],
        mode="lines",
        name="Response vigor",
        line=dict(color="magenta"),
        showlegend=False,
    ),
    row=4,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=np.concatenate([dff["Step"], dff["Step"][::-1]]),
        y=np.concatenate([v_upper, v_lower[::-1]]),
        fill="toself",
        fillcolor="rgba(255,0,255,0.2)",
        line=dict(color="rgba(255,0,255,0)"),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=4,
    col=1,
)

fig.update_yaxes(
    title_text="Response vigor",
    range=[0, 1],
    row=4,
    col=1,
)

fig.add_annotation(
    xref="x domain",
    yref="y domain",
    x=1.5,
    y=0.9,
    showarrow=False,
    text=rf"$I = {I3}$",
    font=dict(size=18),
    row=4,
    col=1,
)

fig.add_vline(x=100, line_dash="dash")
fig.add_hline(y=0.5, row=4, col=1, line_dash="dot")

fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.1,      # move below plots
        xanchor="center",
        x=0.5,
    ),
    margin=dict(l=0, r=0, t=20, b=50),
    height=700,
)

fig.update_annotations(xshift=-350, yshift=-5)

fig.update_xaxes(
    tickmode="linear",
    dtick=25,
)

fig.update_xaxes(
    title_text="Step",
    row=4,
    col=1
)

fig.write_image("plots/plot6.pdf")

lambda_A = 0.8888888955116272, eta = 0.1111111119389534, alpha = 0.4444444477558136, kappa = 1.8888888359069824, lambda_C = 0.25, midpoint_C = 125, w_v_A = 4.44444465637207


# Plot 7

In [13]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.colors
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import gc
import glob
import os
import re

meta_df = pd.read_parquet(
     "008_outburst/output/dashboard/meta_info.parquet"
)

files = glob.glob(
    "008_outburst/output/dashboard/008_outburst_*.parquet"
)

# Determine the available iterations by filename
# Files must follow this convention: 003_value_learning_i.parquet
n_iterations = 0
iterations = []
for f in files:
    match = re.search(r"_(\d+)\.parquet$", f)

    if match:
        iterations += [int(match.group(1))]

if iterations:
    n_iterations = len(iterations)

lambda_A_vals = meta_df["lambda_A_vals"].iloc[0]
eta_vals = meta_df["eta_vals"].iloc[0]
gamma_vals = meta_df["gamma_vals"].iloc[0]
alpha_vals = meta_df["alpha_vals"].iloc[0]
kappa_vals = meta_df["kappa_vals"].iloc[0]
w_v_A_vals = meta_df["w_v_A_vals"].iloc[0]
I_vals = meta_df["I_vals"].iloc[0]
r1_vals = meta_df["r1_vals"].iloc[0]
theta_N_w0_vals = meta_df["theta_N_w0_vals"].iloc[0]
theta_A_w0_vals = meta_df["theta_A_w0_vals"].iloc[0]
theta_A_w1_vals = meta_df["theta_A_w1_vals"].iloc[0]

del meta_df
gc.collect()

# Values to plot
iteration = "expected"
lambda_A = lambda_A_vals[0]  # 0.888
eta = eta_vals[0] # 0.111
gamma = 0
C = 1
alpha = alpha_vals[1]  # 0.444
kappa = kappa_vals[1]  # 1.888
w_v_A = w_v_A_vals[0]
I = 0
r1 = 0
r1_2 = -1
theta_N_w0 = theta_N_w0_vals[0]
theta_A_w0 = theta_A_w0_vals[0]
theta_A_w1 = theta_A_w1_vals[4]  # -20
V0 = 0.5
M_A0 = 0
M_A0_2 = -0.5
M_S0 = 0


print(f"lambda_A = {lambda_A}, eta = {eta}, alpha = {alpha}, kappa = {kappa}, w_v_A = {w_v_A}, I = {I}, r1 = {r1}, theta_N_w0 = {theta_N_w0}, theta_A_w0 = {theta_A_w0}, theta_A_w1 = {theta_A_w1}")


def get_df_by_params(lambda_A, eta, gamma, alpha, kappa, w_v_A, I, C, r1,
                     theta_N_w0, theta_A_w0, theta_A_w1, V0, M_A0, M_S0):
    filters = [
            ("lambda_A",
             "=", lambda_A),
            ("eta",
             "=", eta),
            ("gamma",
             "=", gamma),
            ("alpha",
             "=", alpha),
            ("kappa",
             "=", kappa),
            ("w_v_A",
             "=", w_v_A),
            ("I",
             "=", I),
             ("C",
              "=", C),
             ("r1",
              "=", r1),
             ("theta_N_w0",
              "=", theta_N_w0),
             ("theta_A_w0",
              "=", theta_A_w0),
             ("theta_A_w1",
              "=", theta_A_w1),
             ("V0",
              "=", V0),
             ("M_A0",
              "=", M_A0),
             ("M_S0",
              "=", M_S0),
    ]

    # read only filtered rows and needed columns
    cols_needed = ["Step", "V_mean", "V_std", "M_A_mean", "M_A_std",
                    "M_S_mean", "M_S_std", "C", "v_mean", "v_std", "p_A_mean",
                    "p_A_std"]

    table = pq.read_table(
            "008_outburst/output/dashboard/008_outburst_summary.parquet",
        columns=cols_needed,
        filters=filters
    )

    dff = table.to_pandas().sort_values("Step")

    return dff


fig = make_subplots(
    rows=4,
    cols=4,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=("a", "b", "c", "d",
                    "e", "f", "g", "h",
                    "i", "j", "k", "i",
                    "m", "n", "o", "p"),
    column_titles=(rf"$r_1=0$", rf"$r_1=-1$",
                   rf"$r_1=0$", rf"$r_1=-1$"),
    row_titles=("Value of state", "Frustration/anger", "P[aggressive]", "Response vigor"),
)

# ------------------------------ Column 1  --------------------
dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       alpha=alpha,
                       kappa=kappa,
                       w_v_A=w_v_A,
                       I=I,
                       C=C,
                       r1=r1,
                       theta_N_w0=theta_N_w0,
                       theta_A_w0=theta_A_w0,
                       theta_A_w1=theta_A_w1,
                       V0=V0,
                       M_A0=M_A0,
                       M_S0=M_S0,
                       )

M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]
v_upper = dff["v_mean"] + dff["v_std"]
v_lower = dff["v_mean"] - dff["v_std"]
p_A_upper = dff["p_A_mean"] + dff["p_A_std"]
p_A_lower = dff["p_A_mean"] - dff["p_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["V_mean"],
        mode="lines",
        name="Value of state",
        line=dict(color="blue"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["p_A_mean"],
        mode="lines",
        name="Probability aggressive",
        line=dict(color=plotly.colors.qualitative.Bold[8]),
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["v_mean"],
        mode="lines",
        name="Response vigor",
        line=dict(color="magenta"),
    ),
    row=4,
    col=1,
)

# Row 1
fig.update_yaxes(
    range=[-0.5, 0.5],
    row=1,
    col=1,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.5, 0.5],
    row=1,
    col=2,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.5, 0.5],
    row=1,
    col=3,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.5, 0.5],
    row=1,
    col=4,
    tickmode="linear",
    dtick=0.25,
)

# Row 2
fig.update_yaxes(
    range=[-0.7, 0.7],
    row=2,
    col=1,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.7, 0.7],
    row=2,
    col=2,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.7, 0.7],
    row=2,
    col=3,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.7, 0.7],
    row=2,
    col=4,
    tickmode="linear",
    dtick=0.25,
)

# Row 3
fig.update_yaxes(
    range=[0, 1],
    row=3,
    col=1,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=3,
    col=2,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=3,
    col=3,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=3,
    col=4,
    tickmode="linear",
    dtick=0.25,
)

# Row 4
fig.update_yaxes(
    range=[0, 1],
    row=4,
    col=1,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=4,
    col=2,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=4,
    col=3,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=4,
    col=4,
    tickmode="linear",
    dtick=0.25,
)

# ------------------------------ Column 2  --------------------
dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       alpha=alpha,
                       kappa=kappa,
                       w_v_A=w_v_A,
                       I=I,
                       C=C,
                       r1=r1_2,
                       theta_N_w0=theta_N_w0,
                       theta_A_w0=theta_A_w0,
                       theta_A_w1=theta_A_w1,
                       V0=V0,
                       M_A0=M_A0,
                       M_S0=M_S0,
                       )

M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]
v_upper = dff["v_mean"] + dff["v_std"]
v_lower = dff["v_mean"] - dff["v_std"]
p_A_upper = dff["p_A_mean"] + dff["p_A_std"]
p_A_lower = dff["p_A_mean"] - dff["p_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["V_mean"],
        mode="lines",
        name="Value of state",
        line=dict(color="blue"),
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=2,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["p_A_mean"],
        mode="lines",
        name="Probability aggressive",
        line=dict(color=plotly.colors.qualitative.Bold[8]),
    ),
    row=3,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["v_mean"],
        mode="lines",
        name="Response vigor",
        line=dict(color="magenta"),
    ),
    row=4,
    col=2,
)

# ------------------------------ Column 3  --------------------

dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       alpha=alpha,
                       kappa=kappa,
                       w_v_A=w_v_A,
                       I=I,
                       C=C,
                       r1=r1,
                       theta_N_w0=theta_N_w0,
                       theta_A_w0=theta_A_w0,
                       theta_A_w1=theta_A_w1,
                       V0=V0,
                       M_A0=M_A0_2,
                       M_S0=M_S0,
                       )

M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]
v_upper = dff["v_mean"] + dff["v_std"]
v_lower = dff["v_mean"] - dff["v_std"]
p_A_upper = dff["p_A_mean"] + dff["p_A_std"]
p_A_lower = dff["p_A_mean"] - dff["p_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["V_mean"],
        mode="lines",
        name="Value of state",
        line=dict(color="blue"),
        showlegend=False,
    ),
    row=1,
    col=3,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
        showlegend=False,
    ),
    row=2,
    col=3,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["p_A_mean"],
        mode="lines",
        name="Probability aggressive",
        line=dict(color=plotly.colors.qualitative.Bold[8]),
        showlegend=False,
    ),
    row=3,
    col=3,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["v_mean"],
        mode="lines",
        name="Response vigor",
        line=dict(color="magenta"),
        showlegend=False,
    ),
    row=4,
    col=3,
)

# ------------------------------ Column 4  --------------------
dff = get_df_by_params(lambda_A=lambda_A,
                       eta=eta,
                       gamma=gamma,
                       alpha=alpha,
                       kappa=kappa,
                       w_v_A=w_v_A,
                       I=I,
                       C=C,
                       r1=r1_2,
                       theta_N_w0=theta_N_w0,
                       theta_A_w0=theta_A_w0,
                       theta_A_w1=theta_A_w1,
                       V0=V0,
                       M_A0=M_A0_2,
                       M_S0=M_S0,
                       )

M_A_upper = dff["M_A_mean"] + dff["M_A_std"]
M_A_lower = dff["M_A_mean"] - dff["M_A_std"]
M_S_upper = dff["M_S_mean"] + dff["M_S_std"]
M_S_lower = dff["M_S_mean"] - dff["M_S_std"]
v_upper = dff["v_mean"] + dff["v_std"]
v_lower = dff["v_mean"] - dff["v_std"]
p_A_upper = dff["p_A_mean"] + dff["p_A_std"]
p_A_lower = dff["p_A_mean"] - dff["p_A_std"]

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["V_mean"],
        mode="lines",
        name="Value of state",
        line=dict(color="blue"),
    ),
    row=1,
    col=4,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["M_A_mean"],
        mode="lines",
        name="Frustration/anger",
        line=dict(color="red"),
    ),
    row=2,
    col=4,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["p_A_mean"],
        mode="lines",
        name="Probability aggressive",
        line=dict(color=plotly.colors.qualitative.Bold[8]),
    ),
    row=3,
    col=4,
)

fig.add_trace(
    go.Scatter(
        x=dff["Step"],
        y=dff["v_mean"],
        mode="lines",
        name="Response vigor",
        line=dict(color="magenta"),
    ),
    row=4,
    col=4,
)

# Row 1
fig.update_yaxes(
    range=[-0.5, 0.5],
    row=1,
    col=1,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.5, 0.5],
    row=1,
    col=2,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.5, 0.5],
    row=1,
    col=3,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.5, 0.5],
    row=1,
    col=4,
    tickmode="linear",
    dtick=0.25,
)

# Row 2
fig.update_yaxes(
    range=[-0.7, 0.7],
    row=2,
    col=1,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.7, 0.7],
    row=2,
    col=2,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.7, 0.7],
    row=2,
    col=3,
    tickmode="linear",
    dtick=0.25,
)

fig.update_yaxes(
    range=[-0.7, 0.7],
    row=2,
    col=4,
    tickmode="linear",
    dtick=0.25,
)

# Row 3
fig.update_yaxes(
    range=[0, 1],
    row=3,
    col=1,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=3,
    col=2,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=3,
    col=3,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=3,
    col=4,
    tickmode="linear",
    dtick=0.25,
)

# Row 4
fig.update_yaxes(
    range=[0, 1],
    row=4,
    col=1,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=4,
    col=2,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=4,
    col=3,
    tickmode="linear",
    dtick=0.25,
)
fig.update_yaxes(
    range=[0, 1],
    row=4,
    col=4,
    tickmode="linear",
    dtick=0.25,
)

fig.update_xaxes(
    tickmode="linear",
    dtick=10,
)

fig.update_xaxes(
    title_text="Step",
    row=4,
    col=1,
)

fig.update_xaxes(
    title_text="Step",
    row=4,
    col=2,
)

fig.update_xaxes(
    title_text="Step",
    row=4,
    col=3,
)

fig.update_xaxes(
    title_text="Step",
    row=4,
    col=4,
)

fig.update_layout(
    showlegend=False,
    width=900,
    height=700,
    margin=dict(l=60, r=10, t=70, b=50),
)

# Meta columns annotation
fig.add_annotation(
    text=rf"$M_0^{{(A)}}=0$",
    x=0.35, y=1.13,
    xref="paper", yref="paper",
    showarrow=False,
    font=dict(size=16)
)

fig.add_annotation(
    text=rf"$M_0^{{(A)}}=-0.5$",
    x=0.95, y=1.13,
    xref="paper", yref="paper",
    showarrow=False,
    font=dict(size=16)
)

# Shift annotations around (plot titles etc.)
fig.update_annotations(xshift=-110, yshift=-15)

fig.layout.annotations[20].xshift = -860
fig.layout.annotations[21].xshift = -860
fig.layout.annotations[22].xshift = -860
fig.layout.annotations[23].xshift = -860

fig.layout.annotations[20].yshift = 0
fig.layout.annotations[21].yshift = 0
fig.layout.annotations[22].yshift = 0
fig.layout.annotations[23].yshift = 0

fig.layout.annotations[20].textangle = -90
fig.layout.annotations[21].textangle = -90
fig.layout.annotations[22].textangle = -90
fig.layout.annotations[23].textangle = -90

fig.layout.annotations[20].font.size = 14
fig.layout.annotations[21].font.size = 14
fig.layout.annotations[22].font.size = 14
fig.layout.annotations[23].font.size = 14

# # Shift column titles
fig.layout.annotations[16].xshift = 0
fig.layout.annotations[17].xshift = 0
fig.layout.annotations[18].xshift = 0
fig.layout.annotations[19].xshift = 0
fig.layout.annotations[16].yshift = 15
fig.layout.annotations[17].yshift = 15
fig.layout.annotations[18].yshift = 15
fig.layout.annotations[19].yshift = 15

fig.add_hline(y=0, row=1, col=1)
fig.add_hline(y=0, row=1, col=2)
fig.add_hline(y=0, row=1, col=3)
fig.add_hline(y=0, row=1, col=4)

fig.add_hline(y=0, row=2, col=1)
fig.add_hline(y=0, row=2, col=2)
fig.add_hline(y=0, row=2, col=3)
fig.add_hline(y=0, row=2, col=4)

fig.add_hline(y=0, row=3, col=1)
fig.add_hline(y=0, row=3, col=2)
fig.add_hline(y=0, row=3, col=3)
fig.add_hline(y=0, row=3, col=4)

fig.add_hline(y=0.5, row=4, col=1, line_dash="dot")
fig.add_hline(y=0.5, row=4, col=2, line_dash="dot")
fig.add_hline(y=0.5, row=4, col=3, line_dash="dot")
fig.add_hline(y=0.5, row=4, col=4, line_dash="dot")

fig.write_image("plots/plot7.pdf",
                )

lambda_A = 0.8888888955116272, eta = 0.1111111119389534, alpha = 0.4444444477558136, kappa = 1.8888888359069824, w_v_A = 4.44444465637207, I = 0, r1 = 0, theta_N_w0 = 2.944000005722046, theta_A_w0 = 0, theta_A_w1 = -20
